# Ingestão Batch com CDF

In [ ]:
import uuid
from minio import Minio
from functools import partial
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *


MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM7Class02") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/home/jovyan/work/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/home/jovyan/work/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

## Let's create a new table in the gold layer, from an existing one.

- Schema Enforcement/Evolution

In [ ]:
schema = "id int, data_venda date, produto string, categoria string, quantidade long, preco_unitario double, preco_total double"

In [ ]:
location_raw = f"s3a://staging/batch/clothes"

In [ ]:
# Exemplo de leitura de dados em streaming (ajuste conforme sua fonte e esquema)
staging_df = (
    spark
    .readStream
    .schema(schema)
    .format("csv")
    .option("header", "true")
    .load(location_raw)  # Caminho dos arquivos CSV
)

In [ ]:
staging_df.printSchema()

## INGESTION BATCH DAILY - USING SPARK STREAMING WITH TRIGGER

In [ ]:
table_bronze = "clothes_batch_bronze"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"
location_checkpoint = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}/_checkpoint"

## Creating the BRONZE with CDC enabled (data change feed)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze} (
    id int,
    data_venda date,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_bronze}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
query = (
    staging_df
    .writeStream
    .outputMode("append")  # Adiciona novos registros sem sobrescrever os existentes
    .format("delta")
    .option("path", location_bronze)  # Diretório onde a tabela Delta foi criada
    .option("checkpointLocation", location_checkpoint)  # Local de checkpoint para garantir a consistência
    .trigger(availableNow=True)  # Processa todos os dados disponíveis e encerra o streaming
    .start()
)

print("Streaming of written starting...")

# Aguarda o término do processamento batch
query.awaitTermination()

print("Bronze (staging) table created and data ingested.")

## Exploring more Bronze

In [ ]:
df_bronze = spark.sql(f"SELECT * FROM {DATABASE}.{table_bronze} ORDER BY yearmonthday")

In [ ]:
df_bronze.show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_bronze} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

In [ ]:
df_bronze.count()

## Creating the SILVER with CDC enabled (data change feed)

In [ ]:
table_silver = "clothes_batch_silver"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}"
location_checkpoint_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}/_checkpoint"

In [ ]:
def merge_into_delta_table(cdf_df, batch_id, path):
    """
    Merges CDC data (cdf_df) into the table,
    using the 'id' and 'yearmonthday' fields as the key.
    If the table does not exist, it is created based on the cdf_df structure.
    
    To avoid the error of multiple rows corresponding to the same key, the source DataFrame
    is preprocessed to eliminate duplicates in the key.
    """
    # Tentar carregar a tabela; se não existir, cria uma tabela vazia
    try:
        layer_table = DeltaTable.forPath(spark, path)
    except Exception as e:
        # Tabela não existe. Criando uma tabela vazia
        empty_df = cdf_df.limit(0)
        empty_df.write.format("delta").mode("overwrite").save(path)
        layer_table = DeltaTable.forPath(spark, path)
    
    # Pré-processamento: eliminar duplicatas na chave 'id' e 'yearmonthday'
    deduped_source = cdf_df.dropDuplicates(["id", "yearmonthday"])
    
    # Executa o MERGE: atualiza registros existentes, insere novos e deleta registros ausentes na fonte
    layer_table.alias("t").merge(
        source=deduped_source.alias("s"),
        condition="t.id = s.id AND t.yearmonthday = s.yearmonthday"
    ).whenMatchedDelete(condition="s._change_type = 'delete'") \
        .whenMatchedUpdateAll(condition="s._change_type <> 'delete'") \
        .whenNotMatchedInsertAll(condition="s._change_type <> 'delete'") \
        .execute()

In [ ]:
tables_df = spark.sql(f"SHOW TABLES IN {DATABASE}")
tables_list = [row.tableName for row in tables_df.collect()]

if table_silver in tables_list:
    print(f"A tabela {table_bronze} já existe no database {DATABASE}.")
    # Executa a query sem ORDER BY
    history_df = spark.sql(f"DESCRIBE HISTORY {DATABASE}.{table_bronze}")
    
    # Ordena o DataFrame pela coluna "version" de forma decrescente
    latest_version = history_df.orderBy("version", ascending=False).first()["version"]

    # Define o startingVersion: pega a versão anterior, mas nunca menor que 0
    starting_version = max(int(latest_version) - 1, 0)
    print(f"Lendo mudanças de {table_bronze} do CDC: startingVersion = {starting_version}, latest_version = {latest_version}")
    
    # Se a última versão for maior que o starting_version, usamos o endingVersion; 
    # caso contrário, omitimos essa opção.
    if int(latest_version) > starting_version:
         df_cdf = (
            spark.readStream.format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", starting_version)
                 .option("endingVersion", latest_version)
                 .table(f"{DATABASE}.{table_bronze}")
         )
    else:
         df_cdf = (
            spark.readStream.format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", starting_version)
                 .table(f"{DATABASE}.{table_bronze}")
         )
else:
    # Se a tabela Silver não existir, lê tudo da Bronze (pode ser o primeiro carregamento completo)
    df_cdf = (
        spark.readStream.format("delta")
             .option("readChangeFeed", "true")
             .option("startingVersion", 0)
             .table(f"{DATABASE}.{table_bronze}")
    )

In [ ]:
# df_cdf.show(truncate=False)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_silver} (
    id int,
    data_venda date,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_silver}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
%%time
merge_func = partial(merge_into_delta_table, path=location_silver)

query = (
    df_cdf
    .writeStream
    .format("delta")
    .foreachBatch(merge_func)
    .option("path", location_silver)
    .option("checkpointLocation", location_checkpoint_silver)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Silver (bronze) table created and data ingested.")

## Exploring more Silver

In [ ]:
# Query to count total days of month there are on Bronze
spark.sql(f"""
SELECT
    COUNT(*) as total_silver
FROM {DATABASE}.{table_bronze} 
""").show(truncate=False)

In [ ]:
# Query to count total days of month there are on Silver
spark.sql(f"""
SELECT
    COUNT(*) as total_silver
FROM {DATABASE}.{table_bronze} 
""").show(truncate=False)

## Simulation of Changes in the Bronze Table

On 01/01/2025 there were changes in my transactional database (MySQL). I want to reflect these changes in my data lake

- Update: Change record with id = '1'
- Insert: Insert a new record
- Delete: Remove record with id = '2'

In [ ]:
# UPDATE
spark.sql(f"""
UPDATE {DATABASE}.{table_bronze}
SET quantidade = 2, preco_total = 164.52
WHERE id = 4
""")

In [ ]:
#INSERT
spark.sql(f"""
INSERT INTO {DATABASE}.{table_bronze} (id, data_venda, produto, categoria, quantidade, preco_unitario, preco_total, yearmonthday)
VALUES (
    '21',
    DATE('2025-01-01'),
    'Tenis',
    'Masculino',
    1,
    500.0,
    500.0,
    DATE('2025-01-01')
);
""")

In [ ]:
# DELETE
spark.sql(f"""
DELETE FROM {DATABASE}.{table_bronze}
WHERE id > 15
""")

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_bronze}
WHERE
    id IN(2,4,21)
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_silver}
WHERE
    id IN(2,4,21)
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_bronze}
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_silver}
""").show(truncate=False)

In [ ]:
spark.stop()